# Stage 5: Drug Candidate Validation

**Ancient Drug Discovery Pipeline**

This notebook scores candidates from Stage 4 and filters for drug-likeness:
1. DrugGPT — generate small molecule candidates targeting ERAP2
2. RDKit — Lipinski's Rule of 5, Veber's rules, PAINS filter
3. DiffDock — dock candidates against ERAP2 structure
4. Compare against known inhibitor DG013A / Bestatin

**Success criteria:** Binding affinity < 100nM, pass Lipinski, no toxicity flags

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch requests biopython
# rdkit is pre-installed on Colab — rdkit-pypi package not needed
try:
    from rdkit import Chem
    print("RDKit available (pre-installed)")
except ImportError:
    !pip install -q rdkit
    from rdkit import Chem
    print("RDKit installed")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Download ERAP2 structure if not present
import os, requests

PDB_FILE = "erap2_wt_alphafold.pdb"
if not os.path.exists(PDB_FILE):
    resp = requests.get("https://alphafold.ebi.ac.uk/files/AF-Q6P179-F1-model_v6.pdb")
    resp.raise_for_status()
    with open(PDB_FILE, "w") as f:
        f.write(resp.text)
    print(f"Downloaded ERAP2 structure")
else:
    print(f"Already have {PDB_FILE}")

In [ ]:
# Cell 3: Load DrugGPT for small molecule generation
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading DrugGPT (trained on 1.8M protein-ligand pairs)...")
druggpt_name = "liyuesen/druggpt"

try:
    druggpt_tokenizer = AutoTokenizer.from_pretrained(druggpt_name)
    druggpt_model = AutoModelForCausalLM.from_pretrained(druggpt_name).to(device)
    druggpt_model.eval()
    print(f"DrugGPT loaded: {sum(p.numel() for p in druggpt_model.parameters()) / 1e6:.0f}M params")
except Exception as e:
    print(f"DrugGPT load error: {e}")
    print("Falling back to manual SMILES candidates below.")

In [ ]:
# Cell 4: Comprehensive drug-likeness scoring function
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Draw
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

def score_druglikeness(smiles, name=""):
    """Full drug-likeness assessment."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"name": name, "smiles": smiles, "valid": False}
    
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rotatable = Descriptors.NumRotatableBonds(mol)
    
    # Lipinski's Rule of 5
    lipinski = (mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10)
    
    # Veber's rules (oral bioavailability)
    veber = (tpsa <= 140 and rotatable <= 10)
    
    # PAINS filter
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    catalog = FilterCatalog(params)
    pains_alert = catalog.HasMatch(mol)
    
    return {
        "name": name,
        "smiles": smiles,
        "valid": True,
        "mw": round(mw, 1),
        "logp": round(logp, 2),
        "hbd": hbd,
        "hba": hba,
        "tpsa": round(tpsa, 1),
        "rotatable_bonds": rotatable,
        "lipinski_pass": lipinski,
        "veber_pass": veber,
        "pains_alert": pains_alert,
        "overall_pass": lipinski and veber and not pains_alert,
    }

print("Drug-likeness scoring function ready.")

In [ ]:
# Cell 5: Score reference compounds
import pandas as pd

# Known aminopeptidase inhibitors as baselines
references = [
    ("Bestatin", "CC(O)C(=O)NC(CC1=CC=CC=C1)C(O)=O"),
    ("Tosedostat", "CC(C)CC(NC(=O)C(CC1=CC=CC=C1)OC(=O)C(C)(C)C)B(O)O"),
]

ref_results = [score_druglikeness(smi, name) for name, smi in references]
ref_df = pd.DataFrame(ref_results)

print("Reference Compounds (known aminopeptidase inhibitors)")
print("=" * 70)
for r in ref_results:
    if r["valid"]:
        print(f"\n{r['name']}:")
        print(f"  MW: {r['mw']}  LogP: {r['logp']}  HBD: {r['hbd']}  HBA: {r['hba']}")
        print(f"  TPSA: {r['tpsa']}  RotBonds: {r['rotatable_bonds']}")
        print(f"  Lipinski: {'PASS' if r['lipinski_pass'] else 'FAIL'}")
        print(f"  Veber: {'PASS' if r['veber_pass'] else 'FAIL'}")
        print(f"  PAINS: {'ALERT' if r['pains_alert'] else 'Clean'}")
        print(f"  Overall: {'PASS' if r['overall_pass'] else 'FAIL'}")

In [ ]:
# Cell 6: Generate candidates with DrugGPT (if loaded)
# DrugGPT takes a protein pocket representation and generates SMILES

# Extract ERAP2 active site pocket residues from PDB
pocket_residues = list(range(365, 400))  # Residues surrounding active site

# Read pocket from PDB
with open(PDB_FILE) as f:
    pdb_lines = f.readlines()

pocket_atoms = []
for line in pdb_lines:
    if line.startswith("ATOM"):
        resnum = int(line[22:26].strip())
        if resnum in pocket_residues:
            pocket_atoms.append(line)

print(f"Pocket defined: {len(pocket_atoms)} atoms from residues {min(pocket_residues)}-{max(pocket_residues)}")
print(f"\nTo generate candidates with DrugGPT:")
print(f"  - Feed pocket representation to the model")
print(f"  - Generate 100+ SMILES candidates")
print(f"  - Filter with score_druglikeness()")
print(f"\nAlternatively, dock known aminopeptidase inhibitors from DrugBank")
print(f"against the ERAP2 structure to validate the approach.")

In [ ]:
# Cell 7: Score any generated or manually curated candidates
# Add SMILES strings here as they're generated

candidates = [
    # Add generated candidates here:
    # ("candidate_1", "SMILES_STRING"),
    # ("candidate_2", "SMILES_STRING"),
]

if candidates:
    results = [score_druglikeness(smi, name) for name, smi in candidates]
    df = pd.DataFrame(results)
    
    valid = df[df["valid"] == True]
    passing = valid[valid["overall_pass"] == True]
    
    print(f"Total candidates: {len(df)}")
    print(f"Valid SMILES: {len(valid)}")
    print(f"Pass all filters: {len(passing)}")
    
    if not passing.empty:
        print("\nTop candidates:")
        print(passing[["name", "mw", "logp", "hbd", "hba", "tpsa"]].to_string(index=False))
else:
    print("No candidates yet. Run binder design (Stage 4 notebook) first,")
    print("or generate with DrugGPT in Cell 6 above.")